# Project 2 — Retail Sales SQL & Power BI Analytics

**Ready-to-run Google Colab notebook**

This project uses the public **Sample Superstore** retail dataset to demonstrate:

- SQL database creation
- SQL data cleaning
- KPI analysis
- customer, product and regional analysis
- advanced SQL (CTEs, window functions, `LAG`, `RANK`, `CASE`)
- business visualisation
- Power BI-ready data exports
- DAX measures for the later Power BI dashboard

## Important

Google Colab does not provide a permanent MySQL server by default.  
For a reliable one-click notebook, this project runs the SQL analysis with **SQLite**, which supports the CTEs and window functions used here.

At the end, the notebook also creates **MySQL-ready SQL scripts** for the GitHub version of the project.

For Power BI, the notebook exports a clean CSV and a DAX-measures file. The `.pbix` dashboard will be created later in Power BI Desktop.

### How to run

Choose:

**Runtime → Run all**

The notebook will download the dataset automatically.

## 1. Imports and Project Folders

In [ ]:
from pathlib import Path
import urllib.request
import sqlite3
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_DIR = Path("project_2_outputs")
DATA_DIR = PROJECT_DIR / "data"
SQL_DIR = PROJECT_DIR / "sql"
POWERBI_DIR = PROJECT_DIR / "powerbi"
IMAGE_DIR = PROJECT_DIR / "images"
RESULTS_DIR = PROJECT_DIR / "sql_results"

for folder in [PROJECT_DIR, DATA_DIR, SQL_DIR, POWERBI_DIR, IMAGE_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created.")

## 2. Download the Sample Superstore Dataset

In [ ]:
DATASET_URLS = [
    "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv",
    "https://raw.githubusercontent.com/sumit0072/Superstore-Data-Analysis/main/Sample%20-%20Superstore.csv",
]

RAW_FILE = DATA_DIR / "superstore_raw.csv"

if not RAW_FILE.exists():
    last_error = None

    for url in DATASET_URLS:
        try:
            print(f"Trying dataset source: {url}")
            urllib.request.urlretrieve(url, RAW_FILE)
            print("Dataset downloaded successfully.")
            break
        except Exception as exc:
            last_error = exc
            print(f"Download attempt failed: {exc}")

    if not RAW_FILE.exists():
        raise RuntimeError(
            "The dataset could not be downloaded automatically. "
            "Please upload a Sample Superstore CSV to Colab and rename it "
            "'superstore_raw.csv'."
        ) from last_error
else:
    print("Dataset already exists.")

print("Dataset path:", RAW_FILE)

## 3. Load and Understand the Raw Dataset

In [ ]:
raw_df = pd.read_csv(RAW_FILE)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]}")
display(raw_df.head())

In [ ]:
print("Original columns:")
for col in raw_df.columns:
    print("-", col)

In [ ]:
print("Missing values:")
display(raw_df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))

print("\nDuplicate rows:", raw_df.duplicated().sum())

In [ ]:
display(raw_df.describe(include="all").T)

## 4. Prepare Column Names for SQL

The original data is retained in `superstore_raw.csv`.

For database work, column names are converted to `snake_case`, dates are standardised to `YYYY-MM-DD`, and text values are trimmed. The main cleaning rules are then applied **inside SQL**.

In [ ]:
df = raw_df.copy()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace("-", "_", regex=False)
      .str.replace(" ", "_", regex=False)
)

# Standardise dates before database import.
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")

# Convert to ISO date strings so SQLite date functions work consistently.
df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")
df["ship_date"] = df["ship_date"].dt.strftime("%Y-%m-%d")

# Trim text columns.
text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

print("SQL-friendly columns:")
print(df.columns.tolist())

display(df.head())

## 5. Create the SQL Database

In [ ]:
DB_FILE = PROJECT_DIR / "retail_sales.sqlite"

if DB_FILE.exists():
    DB_FILE.unlink()

conn = sqlite3.connect(DB_FILE)

df.to_sql(
    "raw_orders",
    conn,
    if_exists="replace",
    index=False
)

print(f"Raw table created with {len(df):,} rows.")

### SQL Helper Function

In [ ]:
def run_query(sql, title=None, save_as=None):
    result = pd.read_sql_query(sql, conn)

    if title:
        print(title)

    display(result)

    if save_as:
        result.to_csv(RESULTS_DIR / save_as, index=False)

    return result

## 6. SQL Data Cleaning

The SQL cleaning stage:

- removes exact duplicate transaction lines
- requires valid order/customer/product IDs
- removes invalid dates
- keeps only positive sales and quantities
- keeps discount values between 0 and 1
- retains negative profit because a loss-making transaction is valid business information
- creates year, quarter, month, shipping-days and profit-margin features

In [ ]:
SQLITE_CLEANING_SQL = r'''
DROP TABLE IF EXISTS orders_clean;

CREATE TABLE orders_clean AS
WITH deduped AS (
    SELECT DISTINCT
        order_id,
        order_date,
        ship_date,
        TRIM(ship_mode) AS ship_mode,
        customer_id,
        TRIM(customer_name) AS customer_name,
        TRIM(segment) AS segment,
        TRIM(country) AS country,
        TRIM(city) AS city,
        TRIM(state) AS state,
        postal_code,
        TRIM(region) AS region,
        product_id,
        TRIM(category) AS category,
        TRIM(sub_category) AS sub_category,
        TRIM(product_name) AS product_name,
        sales,
        quantity,
        discount,
        profit
    FROM raw_orders
)
SELECT
    *,
    CAST(strftime('%Y', order_date) AS INTEGER) AS order_year,
    'Q' || (
        CAST(
            (CAST(strftime('%m', order_date) AS INTEGER) - 1) / 3
            AS INTEGER
        ) + 1
    ) AS order_quarter,
    strftime('%Y-%m', order_date) AS order_month,
    CAST(
        julianday(ship_date) - julianday(order_date)
        AS INTEGER
    ) AS shipping_days,
    ROUND(
        100.0 * profit / NULLIF(sales, 0),
        2
    ) AS profit_margin_pct
FROM deduped
WHERE
    order_id IS NOT NULL
    AND TRIM(order_id) <> ''
    AND customer_id IS NOT NULL
    AND TRIM(customer_id) <> ''
    AND product_id IS NOT NULL
    AND TRIM(product_id) <> ''
    AND order_date IS NOT NULL
    AND ship_date IS NOT NULL
    AND sales > 0
    AND quantity > 0
    AND discount BETWEEN 0 AND 1;
'''

conn.executescript(SQLITE_CLEANING_SQL)

clean_count = pd.read_sql_query(
    "SELECT COUNT(*) AS rows_after_cleaning FROM orders_clean",
    conn
)

display(clean_count)

In [ ]:
clean_df = pd.read_sql_query("SELECT * FROM orders_clean", conn)

CLEAN_FILE = DATA_DIR / "superstore_cleaned.csv"
POWERBI_FILE = POWERBI_DIR / "superstore_powerbi.csv"

clean_df.to_csv(CLEAN_FILE, index=False)
clean_df.to_csv(POWERBI_FILE, index=False)

print(f"Cleaned rows: {len(clean_df):,}")
print(f"Cleaned CSV: {CLEAN_FILE}")
print(f"Power BI CSV: {POWERBI_FILE}")
display(clean_df.head())

## 7. Data Quality Validation

In [ ]:
validation = pd.DataFrame({
    "Check": [
        "Raw rows",
        "Clean rows",
        "Exact duplicates in clean data",
        "Missing order IDs",
        "Missing customer IDs",
        "Missing product IDs",
        "Non-positive sales",
        "Non-positive quantities",
        "Invalid discounts",
    ],
    "Value": [
        len(df),
        len(clean_df),
        clean_df.duplicated().sum(),
        clean_df["order_id"].isna().sum(),
        clean_df["customer_id"].isna().sum(),
        clean_df["product_id"].isna().sum(),
        (clean_df["sales"] <= 0).sum(),
        (clean_df["quantity"] <= 0).sum(),
        ((clean_df["discount"] < 0) | (clean_df["discount"] > 1)).sum(),
    ]
})

display(validation)

# SQL Business Analysis

## Question 1 — What are total sales, total profit and overall profit margin?

In [ ]:
q1 = run_query(
    '''
    SELECT
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean;
    ''',
    "Q1 — Overall Sales, Profit and Profit Margin",
    "q01_overall_kpis.csv"
)

## Question 2 — How many unique orders and customers are in the dataset?

In [ ]:
q2 = run_query(
    '''
    SELECT
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT customer_id) AS total_customers,
        SUM(quantity) AS total_quantity
    FROM orders_clean;
    ''',
    "Q2 — Orders, Customers and Quantity",
    "q02_orders_customers.csv"
)

## Question 3 — What is the average order value?

In [ ]:
q3 = run_query(
    '''
    SELECT
        ROUND(
            SUM(sales) / NULLIF(COUNT(DISTINCT order_id), 0),
            2
        ) AS average_order_value
    FROM orders_clean;
    ''',
    "Q3 — Average Order Value",
    "q03_average_order_value.csv"
)

## Question 4 — Which 10 products generate the highest sales?

In [ ]:
q4 = run_query(
    '''
    SELECT
        product_id,
        product_name,
        ROUND(SUM(sales), 2) AS total_sales,
        SUM(quantity) AS units_sold
    FROM orders_clean
    GROUP BY product_id, product_name
    ORDER BY total_sales DESC
    LIMIT 10;
    ''',
    "Q4 — Top 10 Products by Sales",
    "q04_top_products_sales.csv"
)

## Question 5 — Which 10 products generate the highest profit?

In [ ]:
q5 = run_query(
    '''
    SELECT
        product_id,
        product_name,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(SUM(sales), 2) AS total_sales
    FROM orders_clean
    GROUP BY product_id, product_name
    ORDER BY total_profit DESC
    LIMIT 10;
    ''',
    "Q5 — Top 10 Products by Profit",
    "q05_top_products_profit.csv"
)

## Question 6 — Which products have high sales but negative profit?

In [ ]:
q6 = run_query(
    '''
    SELECT
        product_id,
        product_name,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean
    GROUP BY product_id, product_name
    HAVING SUM(profit) < 0
    ORDER BY total_sales DESC
    LIMIT 15;
    ''',
    "Q6 — High-Sales Loss-Making Products",
    "q06_loss_making_products.csv"
)

## Question 7 — Which categories and sub-categories perform best?

In [ ]:
q7 = run_query(
    '''
    SELECT
        category,
        sub_category,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean
    GROUP BY category, sub_category
    ORDER BY total_sales DESC;
    ''',
    "Q7 — Category and Sub-Category Performance",
    "q07_category_subcategory.csv"
)

## Question 8 — Which customer segment generates the most sales and profit?

In [ ]:
q8 = run_query(
    '''
    SELECT
        segment,
        COUNT(DISTINCT customer_id) AS customers,
        COUNT(DISTINCT order_id) AS orders,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean
    GROUP BY segment
    ORDER BY total_sales DESC;
    ''',
    "Q8 — Customer Segment Performance",
    "q08_customer_segments.csv"
)

## Question 9 — Who are the top 10 customers by lifetime sales?

In [ ]:
q9 = run_query(
    '''
    SELECT
        customer_id,
        customer_name,
        segment,
        COUNT(DISTINCT order_id) AS orders,
        ROUND(SUM(sales), 2) AS lifetime_sales,
        ROUND(SUM(profit), 2) AS lifetime_profit
    FROM orders_clean
    GROUP BY customer_id, customer_name, segment
    ORDER BY lifetime_sales DESC
    LIMIT 10;
    ''',
    "Q9 — Top 10 Customers by Lifetime Sales",
    "q09_top_customers_sales.csv"
)

## Question 10 — Which customers place the most orders?

In [ ]:
q10 = run_query(
    '''
    SELECT
        customer_id,
        customer_name,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(
            SUM(sales) / NULLIF(COUNT(DISTINCT order_id), 0),
            2
        ) AS average_order_value
    FROM orders_clean
    GROUP BY customer_id, customer_name
    ORDER BY total_orders DESC, total_sales DESC
    LIMIT 15;
    ''',
    "Q10 — Customers with Most Orders",
    "q10_customer_order_frequency.csv"
)

## Question 11 — Which regions generate the highest and lowest sales?

In [ ]:
q11 = run_query(
    '''
    SELECT
        region,
        COUNT(DISTINCT order_id) AS orders,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit
    FROM orders_clean
    GROUP BY region
    ORDER BY total_sales DESC;
    ''',
    "Q11 — Regional Sales",
    "q11_regional_sales.csv"
)

## Question 12 — Which regions have the best and worst profit margin?

In [ ]:
q12 = run_query(
    '''
    SELECT
        region,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean
    GROUP BY region
    ORDER BY profit_margin_pct DESC;
    ''',
    "Q12 — Regional Profit Margin",
    "q12_regional_profit_margin.csv"
)

## Question 13 — Which states or cities are loss-making?

In [ ]:
q13 = run_query(
    '''
    SELECT
        state,
        city,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean
    GROUP BY state, city
    HAVING SUM(profit) < 0
    ORDER BY total_profit ASC
    LIMIT 20;
    ''',
    "Q13 — Loss-Making States/Cities",
    "q13_loss_making_locations.csv"
)

## Question 14 — How do monthly sales and profit change over time?

In [ ]:
q14 = run_query(
    '''
    SELECT
        order_month,
        ROUND(SUM(sales), 2) AS monthly_sales,
        ROUND(SUM(profit), 2) AS monthly_profit,
        COUNT(DISTINCT order_id) AS orders
    FROM orders_clean
    GROUP BY order_month
    ORDER BY order_month;
    ''',
    "Q14 — Monthly Sales and Profit",
    "q14_monthly_sales_profit.csv"
)

## Question 15 — What is the month-over-month sales growth rate?

In [ ]:
q15 = run_query(
    '''
    WITH monthly AS (
        SELECT
            order_month,
            SUM(sales) AS monthly_sales
        FROM orders_clean
        GROUP BY order_month
    ),
    previous_month AS (
        SELECT
            order_month,
            monthly_sales,
            LAG(monthly_sales) OVER (
                ORDER BY order_month
            ) AS previous_month_sales
        FROM monthly
    )
    SELECT
        order_month,
        ROUND(monthly_sales, 2) AS monthly_sales,
        ROUND(previous_month_sales, 2) AS previous_month_sales,
        ROUND(
            100.0 * (
                monthly_sales - previous_month_sales
            ) / NULLIF(previous_month_sales, 0),
            2
        ) AS mom_growth_pct
    FROM previous_month
    ORDER BY order_month;
    ''',
    "Q15 — Month-over-Month Sales Growth",
    "q15_mom_growth.csv"
)

## Question 16 — Which month has the highest average profit per order?

In [ ]:
q16 = run_query(
    '''
    WITH order_totals AS (
        SELECT
            order_id,
            order_month,
            SUM(profit) AS order_profit
        FROM orders_clean
        GROUP BY order_id, order_month
    )
    SELECT
        order_month,
        ROUND(AVG(order_profit), 2) AS avg_profit_per_order,
        COUNT(*) AS orders
    FROM order_totals
    GROUP BY order_month
    ORDER BY avg_profit_per_order DESC
    LIMIT 12;
    ''',
    "Q16 — Highest Average Profit per Order by Month",
    "q16_avg_profit_per_order.csv"
)

## Question 17 — How does discount level affect profitability?

In [ ]:
q17 = run_query(
    '''
    SELECT
        CASE
            WHEN discount = 0 THEN 'No Discount'
            WHEN discount <= 0.10 THEN '1-10%'
            WHEN discount <= 0.20 THEN '11-20%'
            WHEN discount <= 0.30 THEN '21-30%'
            WHEN discount <= 0.50 THEN '31-50%'
            ELSE 'Above 50%'
        END AS discount_band,
        COUNT(*) AS transaction_lines,
        ROUND(AVG(discount) * 100, 2) AS avg_discount_pct,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean
    GROUP BY discount_band
    ORDER BY avg_discount_pct;
    ''',
    "Q17 — Discount and Profitability",
    "q17_discount_profitability.csv"
)

## Question 18 — What percentage of orders are profitable vs loss-making?

In [ ]:
q18 = run_query(
    '''
    WITH order_profit AS (
        SELECT
            order_id,
            SUM(profit) AS total_order_profit
        FROM orders_clean
        GROUP BY order_id
    ),
    classified AS (
        SELECT
            CASE
                WHEN total_order_profit > 0 THEN 'Profitable'
                WHEN total_order_profit < 0 THEN 'Loss-Making'
                ELSE 'Break-Even'
            END AS order_status
        FROM order_profit
    )
    SELECT
        order_status,
        COUNT(*) AS orders,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage_of_orders
    FROM classified
    GROUP BY order_status
    ORDER BY orders DESC;
    ''',
    "Q18 — Profitable vs Loss-Making Orders",
    "q18_order_profitability.csv"
)

## Question 19 — Rank products within each category by sales.

In [ ]:
q19 = run_query(
    '''
    WITH product_sales AS (
        SELECT
            category,
            product_id,
            product_name,
            SUM(sales) AS total_sales
        FROM orders_clean
        GROUP BY category, product_id, product_name
    ),
    ranked AS (
        SELECT
            category,
            product_id,
            product_name,
            total_sales,
            RANK() OVER (
                PARTITION BY category
                ORDER BY total_sales DESC
            ) AS sales_rank
        FROM product_sales
    )
    SELECT
        category,
        product_id,
        product_name,
        ROUND(total_sales, 2) AS total_sales,
        sales_rank
    FROM ranked
    WHERE sales_rank <= 5
    ORDER BY category, sales_rank;
    ''',
    "Q19 — Top Products Within Each Category",
    "q19_ranked_products.csv"
)

## Question 20 — Which shipping mode balances delivery speed and profitability?

In [ ]:
q20 = run_query(
    '''
    SELECT
        ship_mode,
        COUNT(DISTINCT order_id) AS orders,
        ROUND(AVG(shipping_days), 2) AS avg_shipping_days,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(
            100.0 * SUM(profit) / NULLIF(SUM(sales), 0),
            2
        ) AS profit_margin_pct
    FROM orders_clean
    GROUP BY ship_mode
    ORDER BY profit_margin_pct DESC;
    ''',
    "Q20 — Shipping Mode Performance",
    "q20_shipping_mode.csv"
)

# 8. Business Visualisations

### Monthly Sales and Profit Trend

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(q14["order_month"], q14["monthly_sales"], marker="o", label="Sales")
plt.plot(q14["order_month"], q14["monthly_profit"], marker="o", label="Profit")
plt.title("Monthly Sales and Profit Trend")
plt.xlabel("Month")
plt.ylabel("Amount")
plt.xticks(rotation=70)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(IMAGE_DIR / "monthly_sales_profit.png", dpi=150, bbox_inches="tight")
plt.show()

### Top 10 Products by Sales

In [ ]:
plot_q4 = q4.sort_values("total_sales")

plt.figure(figsize=(11, 6))
plt.barh(plot_q4["product_name"], plot_q4["total_sales"])
plt.title("Top 10 Products by Sales")
plt.xlabel("Sales")
plt.ylabel("Product")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "top_products_sales.png", dpi=150, bbox_inches="tight")
plt.show()

### Regional Sales and Profit

In [ ]:
x = np.arange(len(q11["region"]))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, q11["total_sales"], width, label="Sales")
plt.bar(x + width/2, q11["total_profit"], width, label="Profit")
plt.xticks(x, q11["region"])
plt.title("Regional Sales and Profit")
plt.ylabel("Amount")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGE_DIR / "regional_sales_profit.png", dpi=150, bbox_inches="tight")
plt.show()

### Customer Segment Performance

In [ ]:
x = np.arange(len(q8["segment"]))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, q8["total_sales"], width, label="Sales")
plt.bar(x + width/2, q8["total_profit"], width, label="Profit")
plt.xticks(x, q8["segment"])
plt.title("Customer Segment Sales and Profit")
plt.ylabel("Amount")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGE_DIR / "customer_segment_performance.png", dpi=150, bbox_inches="tight")
plt.show()

### Discount vs Profitability

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(q17["discount_band"], q17["profit_margin_pct"])
plt.title("Profit Margin by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(IMAGE_DIR / "discount_profitability.png", dpi=150, bbox_inches="tight")
plt.show()

# 9. Project Summary

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Profit",
        "Profit Margin %",
        "Unique Orders",
        "Unique Customers",
        "Average Order Value",
        "Best Sales Region",
        "Best Sales Category",
    ],
    "Value": [
        float(q1.loc[0, "total_sales"]),
        float(q1.loc[0, "total_profit"]),
        float(q1.loc[0, "profit_margin_pct"]),
        int(q2.loc[0, "total_orders"]),
        int(q2.loc[0, "total_customers"]),
        float(q3.loc[0, "average_order_value"]),
        q11.iloc[0]["region"],
        q7.groupby("category")["total_sales"].sum().idxmax(),
    ]
})

display(summary)
summary.to_csv(PROJECT_DIR / "project_summary.csv", index=False)

# 10. Power BI Preparation

The exported file:

`project_2_outputs/powerbi/superstore_powerbi.csv`

is the clean dataset to import into Power BI Desktop.

The notebook also generates a DAX-measures file below.

In [ ]:
DAX_TEXT = r'''
POWER BI SETUP
==============

Recommended table name:
Superstore

1) Create a Date table:

DateTable =
CALENDAR(
    MIN(Superstore[order_date]),
    MAX(Superstore[order_date])
)

Year = YEAR(DateTable[Date])
Month Number = MONTH(DateTable[Date])
Month = FORMAT(DateTable[Date], "MMMM")
Year Month = FORMAT(DateTable[Date], "YYYY-MM")

Create a relationship:
DateTable[Date] -> Superstore[order_date]

CORE MEASURES
=============

Total Sales =
SUM(Superstore[sales])

Total Profit =
SUM(Superstore[profit])

Total Orders =
DISTINCTCOUNT(Superstore[order_id])

Total Customers =
DISTINCTCOUNT(Superstore[customer_id])

Total Quantity =
SUM(Superstore[quantity])

Profit Margin % =
DIVIDE(
    [Total Profit],
    [Total Sales],
    0
)

Average Order Value =
DIVIDE(
    [Total Sales],
    [Total Orders],
    0
)

Average Discount % =
AVERAGE(Superstore[discount])

Sales Previous Month =
CALCULATE(
    [Total Sales],
    DATEADD(DateTable[Date], -1, MONTH)
)

Month-over-Month Growth % =
DIVIDE(
    [Total Sales] - [Sales Previous Month],
    [Sales Previous Month],
    0
)
'''

DAX_FILE = POWERBI_DIR / "powerbi_dax_measures.txt"
DAX_FILE.write_text(DAX_TEXT, encoding="utf-8")

print(DAX_TEXT)
print("\nSaved:", DAX_FILE)

## Recommended Power BI Dashboard Pages

### Page 1 — Executive Overview
- Total Sales
- Total Profit
- Profit Margin
- Total Orders
- Total Customers
- Average Order Value
- Monthly Sales & Profit Trend
- Date, Region and Category slicers

### Page 2 — Product Performance
- Category and Sub-Category performance
- Top products by sales
- Top products by profit
- Loss-making products
- Sales vs Profit

### Page 3 — Customer & Regional Analysis
- Segment performance
- Top customers
- Regional sales and profit
- State/City analysis
- Map visual where appropriate

### Page 4 — Discount & Profitability
- Discount bands
- Profit margins
- Loss-making products/locations
- Profitability recommendations

# 11. Create MySQL-Ready SQL Files for GitHub

In [ ]:
MYSQL_DATABASE_SETUP = r'''
CREATE DATABASE IF NOT EXISTS retail_sales_analytics;
USE retail_sales_analytics;

DROP TABLE IF EXISTS orders;

CREATE TABLE orders (
    order_id VARCHAR(50),
    order_date DATE,
    ship_date DATE,
    ship_mode VARCHAR(50),
    customer_id VARCHAR(50),
    customer_name VARCHAR(150),
    segment VARCHAR(50),
    country VARCHAR(100),
    city VARCHAR(100),
    state VARCHAR(100),
    postal_code VARCHAR(20),
    region VARCHAR(50),
    product_id VARCHAR(50),
    category VARCHAR(100),
    sub_category VARCHAR(100),
    product_name VARCHAR(255),
    sales DECIMAL(14,4),
    quantity INT,
    discount DECIMAL(8,4),
    profit DECIMAL(14,4),
    order_year INT,
    order_quarter VARCHAR(10),
    order_month VARCHAR(10),
    shipping_days INT,
    profit_margin_pct DECIMAL(12,4)
);

CREATE INDEX idx_orders_order_id ON orders(order_id);
CREATE INDEX idx_orders_customer_id ON orders(customer_id);
CREATE INDEX idx_orders_product_id ON orders(product_id);
CREATE INDEX idx_orders_order_date ON orders(order_date);
CREATE INDEX idx_orders_region ON orders(region);

-- Import project_2_outputs/data/superstore_cleaned.csv
-- using MySQL Workbench's Table Data Import Wizard.
'''

(SQL_DIR / "database_setup.sql").write_text(
    MYSQL_DATABASE_SETUP,
    encoding="utf-8"
)

print("Created:", SQL_DIR / "database_setup.sql")

In [ ]:
MYSQL_CLEANING_QUERIES = r'''
-- Data-quality checks to run after importing into MySQL.

SELECT COUNT(*) AS total_rows
FROM orders;

SELECT
    SUM(order_id IS NULL OR order_id = '') AS missing_order_ids,
    SUM(customer_id IS NULL OR customer_id = '') AS missing_customer_ids,
    SUM(product_id IS NULL OR product_id = '') AS missing_product_ids
FROM orders;

SELECT
    SUM(sales <= 0) AS invalid_sales,
    SUM(quantity <= 0) AS invalid_quantities,
    SUM(discount < 0 OR discount > 1) AS invalid_discounts
FROM orders;

SELECT
    order_id,
    product_id,
    customer_id,
    order_date,
    COUNT(*) AS duplicate_count
FROM orders
GROUP BY
    order_id,
    product_id,
    customer_id,
    order_date,
    sales,
    quantity,
    discount,
    profit
HAVING COUNT(*) > 1;
'''

(SQL_DIR / "cleaning_queries.sql").write_text(
    MYSQL_CLEANING_QUERIES,
    encoding="utf-8"
)

print("Created:", SQL_DIR / "cleaning_queries.sql")

In [ ]:
MYSQL_ANALYSIS_QUERIES = r'''
USE retail_sales_analytics;

-- Q1: Total sales, profit and profit margin
SELECT
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders;

-- Q2: Orders and customers
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers,
    SUM(quantity) AS total_quantity
FROM orders;

-- Q3: Average order value
SELECT
    ROUND(
        SUM(sales) / NULLIF(COUNT(DISTINCT order_id), 0),
        2
    ) AS average_order_value
FROM orders;

-- Q4: Top 10 products by sales
SELECT
    product_id,
    product_name,
    ROUND(SUM(sales), 2) AS total_sales,
    SUM(quantity) AS units_sold
FROM orders
GROUP BY product_id, product_name
ORDER BY total_sales DESC
LIMIT 10;

-- Q5: Top 10 products by profit
SELECT
    product_id,
    product_name,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(sales), 2) AS total_sales
FROM orders
GROUP BY product_id, product_name
ORDER BY total_profit DESC
LIMIT 10;

-- Q6: High-sales loss-making products
SELECT
    product_id,
    product_name,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders
GROUP BY product_id, product_name
HAVING SUM(profit) < 0
ORDER BY total_sales DESC
LIMIT 15;

-- Q7: Category and sub-category performance
SELECT
    category,
    sub_category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders
GROUP BY category, sub_category
ORDER BY total_sales DESC;

-- Q8: Customer segment performance
SELECT
    segment,
    COUNT(DISTINCT customer_id) AS customers,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders
GROUP BY segment
ORDER BY total_sales DESC;

-- Q9: Top customers by lifetime sales
SELECT
    customer_id,
    customer_name,
    segment,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(sales), 2) AS lifetime_sales,
    ROUND(SUM(profit), 2) AS lifetime_profit
FROM orders
GROUP BY customer_id, customer_name, segment
ORDER BY lifetime_sales DESC
LIMIT 10;

-- Q10: Customers with most orders
SELECT
    customer_id,
    customer_name,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(sales) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS average_order_value
FROM orders
GROUP BY customer_id, customer_name
ORDER BY total_orders DESC, total_sales DESC
LIMIT 15;

-- Q11: Regional sales
SELECT
    region,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM orders
GROUP BY region
ORDER BY total_sales DESC;

-- Q12: Regional profit margin
SELECT
    region,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders
GROUP BY region
ORDER BY profit_margin_pct DESC;

-- Q13: Loss-making locations
SELECT
    state,
    city,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders
GROUP BY state, city
HAVING SUM(profit) < 0
ORDER BY total_profit ASC
LIMIT 20;

-- Q14: Monthly sales and profit
SELECT
    order_month,
    ROUND(SUM(sales), 2) AS monthly_sales,
    ROUND(SUM(profit), 2) AS monthly_profit,
    COUNT(DISTINCT order_id) AS orders
FROM orders
GROUP BY order_month
ORDER BY order_month;

-- Q15: Month-over-month growth
WITH monthly AS (
    SELECT
        order_month,
        SUM(sales) AS monthly_sales
    FROM orders
    GROUP BY order_month
),
previous_month AS (
    SELECT
        order_month,
        monthly_sales,
        LAG(monthly_sales) OVER (ORDER BY order_month) AS previous_month_sales
    FROM monthly
)
SELECT
    order_month,
    ROUND(monthly_sales, 2) AS monthly_sales,
    ROUND(previous_month_sales, 2) AS previous_month_sales,
    ROUND(
        100 * (monthly_sales - previous_month_sales)
        / NULLIF(previous_month_sales, 0),
        2
    ) AS mom_growth_pct
FROM previous_month
ORDER BY order_month;

-- Q16: Highest average profit per order by month
WITH order_totals AS (
    SELECT
        order_id,
        order_month,
        SUM(profit) AS order_profit
    FROM orders
    GROUP BY order_id, order_month
)
SELECT
    order_month,
    ROUND(AVG(order_profit), 2) AS avg_profit_per_order,
    COUNT(*) AS orders
FROM order_totals
GROUP BY order_month
ORDER BY avg_profit_per_order DESC
LIMIT 12;

-- Q17: Discount and profitability
SELECT
    CASE
        WHEN discount = 0 THEN 'No Discount'
        WHEN discount <= 0.10 THEN '1-10%'
        WHEN discount <= 0.20 THEN '11-20%'
        WHEN discount <= 0.30 THEN '21-30%'
        WHEN discount <= 0.50 THEN '31-50%'
        ELSE 'Above 50%'
    END AS discount_band,
    COUNT(*) AS transaction_lines,
    ROUND(AVG(discount) * 100, 2) AS avg_discount_pct,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders
GROUP BY discount_band
ORDER BY avg_discount_pct;

-- Q18: Profitable vs loss-making orders
WITH order_profit AS (
    SELECT
        order_id,
        SUM(profit) AS total_order_profit
    FROM orders
    GROUP BY order_id
),
classified AS (
    SELECT
        CASE
            WHEN total_order_profit > 0 THEN 'Profitable'
            WHEN total_order_profit < 0 THEN 'Loss-Making'
            ELSE 'Break-Even'
        END AS order_status
    FROM order_profit
)
SELECT
    order_status,
    COUNT(*) AS orders,
    ROUND(
        100 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_orders
FROM classified
GROUP BY order_status
ORDER BY orders DESC;

-- Q19: Rank products within category
WITH product_sales AS (
    SELECT
        category,
        product_id,
        product_name,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY category, product_id, product_name
),
ranked AS (
    SELECT
        category,
        product_id,
        product_name,
        total_sales,
        RANK() OVER (
            PARTITION BY category
            ORDER BY total_sales DESC
        ) AS sales_rank
    FROM product_sales
)
SELECT
    category,
    product_id,
    product_name,
    ROUND(total_sales, 2) AS total_sales,
    sales_rank
FROM ranked
WHERE sales_rank <= 5
ORDER BY category, sales_rank;

-- Q20: Shipping mode performance
SELECT
    ship_mode,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(AVG(shipping_days), 2) AS avg_shipping_days,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS profit_margin_pct
FROM orders
GROUP BY ship_mode
ORDER BY profit_margin_pct DESC;
'''

(SQL_DIR / "analysis_queries.sql").write_text(
    MYSQL_ANALYSIS_QUERIES,
    encoding="utf-8"
)

print("Created:", SQL_DIR / "analysis_queries.sql")

# 12. Export a Short Business Findings File

In [ ]:
def money(value):
    return f"${value:,.2f}"

best_region = q11.iloc[0]["region"]
best_segment = q8.iloc[0]["segment"]
top_product = q4.iloc[0]["product_name"]
worst_discount_band = q17.sort_values("profit_margin_pct").iloc[0]["discount_band"]

FINDINGS = f'''
RETAIL SALES & CUSTOMER ANALYTICS — KEY FINDINGS
================================================

Total Sales: {money(float(q1.loc[0, "total_sales"]))}
Total Profit: {money(float(q1.loc[0, "total_profit"]))}
Profit Margin: {float(q1.loc[0, "profit_margin_pct"]):.2f}%
Unique Orders: {int(q2.loc[0, "total_orders"]):,}
Unique Customers: {int(q2.loc[0, "total_customers"]):,}
Average Order Value: {money(float(q3.loc[0, "average_order_value"]))}

Best Region by Sales: {best_region}
Best Customer Segment by Sales: {best_segment}
Top Product by Sales: {top_product}
Weakest Discount Band by Profit Margin: {worst_discount_band}

BUSINESS RECOMMENDATIONS
========================

1. Protect high-performing products and categories by monitoring stock availability and demand.
2. Review loss-making products with high sales to identify pricing, discounting or cost problems.
3. Focus customer-retention activity on high-value and repeat customers.
4. Use regional profitability, not sales alone, when allocating marketing and sales resources.
5. Review aggressive discount bands that produce weak or negative profit margins.
6. Monitor month-over-month growth and investigate unusual declines quickly.
7. Use the Power BI dashboard as an ongoing management tool rather than a one-time report.
'''

FINDINGS_FILE = PROJECT_DIR / "business_findings.txt"
FINDINGS_FILE.write_text(FINDINGS, encoding="utf-8")

print(FINDINGS)

# 13. Create a ZIP of All Project Outputs

In [ ]:
conn.close()

ZIP_OUTPUT = Path("Hassan_Project_2_Colab_Outputs.zip")

if ZIP_OUTPUT.exists():
    ZIP_OUTPUT.unlink()

shutil.make_archive(
    ZIP_OUTPUT.with_suffix("").as_posix(),
    "zip",
    PROJECT_DIR
)

print("PROJECT COMPLETE")
print("=" * 60)
print("Download this file from the Colab Files panel:")
print(ZIP_OUTPUT)
print("\nIt contains:")
print("- cleaned and Power BI-ready datasets")
print("- SQLite database")
print("- 20 SQL result CSV files")
print("- MySQL database/setup scripts")
print("- MySQL analysis queries")
print("- Power BI DAX measures")
print("- charts")
print("- business findings")

# Project Complete

After **Runtime → Run all** finishes successfully, download:

`Hassan_Project_2_Colab_Outputs.zip`

from the Colab Files panel.

We will use those files to build the GitHub project next.

The remaining task that must be completed outside Colab is the actual **Power BI `.pbix` dashboard**.